In this notebook, we will investigate the ART matrix and see if we can find a pressure domain PU matrix that matches the ART energy matrix, $\mathbf{S}$ exactly. The ART matrix is column-stochastic, i.e, 
$$
\begin{aligned}
\sum_h S_{h\to i \to j} &= 1 \quad \forall\, i,j \quad \text{(lossless)}, \\
\mathbf{1}^\top \mathbf{S} &= \mathbf{1}^\top.
\end{aligned}
$$

Now, we want to find a PU matrix, $\mathbf{H}(e^{j\omega})$ such that
$$\frac{1}{2 \pi} \int_0^{2\pi}|H_{ij}(e^{j\omega})|^2 d\omega = S_{ij}, \quad \mathbf{H}(e^{j\omega})\mathbf{H}(e^{-j\omega})^\top = \mathbf{I}$$

Note that $|\mathbf{H}(e^{j\omega})|^{\odot 2}$ is doubly stochastic. However, $\mathbf{S}$ is only column-stochastic. For the above equivalence to hold, $\mathbf{S}$ must be made doubly stochastic using the Sinkhorn-Knopp algorthm. However, for this $\mathbf{S}$ requires total support but that is unlikely due to the sparsity of the matrix.

1. First, we prune the matrix $\mathbf{S}$ to remove connections of patches that are not visible to each other. For an $\mathbf{S} \in \mathbb{R}^{+(N \times N)}$, if $k$ patches are not visible to each other then the matrix is pruned to be of size, $\mathbf{S}_{p} \in \mathbb{R}^{+(N-k) \times (N-k)}$.
2.  We run Sinkhorn-Knopp on the pruned matrix to get a doubly stochastic matrix, $\hat{\mathbf{S}}_p, \text{ s.t. } \mathbf{1}^\top\hat{\mathbf{S}}_p = \mathbf{1}^T,  \hat{\mathbf{S}}_p \mathbf{1} = \mathbf{1}.$
3.  We find the FIR PU matrix, $\mathbf{H}(z)$, s.t, $\int_{0}^{2\pi} |\mathbf{H}(e^{j\omega})|^{\odot 2}d \omega = \hat{\mathbf{S}}_p$.
4.  We operate TD-ART in the pressure domain with the FIR PU matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
import os
from pathlib import Path
from loguru import logger
import SDNPy.utils.MatrixMath as mm

In [ ]:
patch_area = 2.0
# environment_name = f'ERTD_generated_patch_area={patch_area:.1f}'
environment_name = 'ERTD_1_patch_per_wall'
environment_folder = os.path.join('..', 'environment', environment_name)
fig_path = Path('../../../Figures/ART/ERTD')
fig_path.mkdir(parents=True, exist_ok=True)

#### Read ART patching matrix and reflection matrix

In [ ]:
# Read the .mtx file
art_matrix_diffuse = scipy.io.mmread(f'{environment_folder}/ART_kernel_diffuse.mtx')
art_matrix_specular = scipy.io.mmread(f'{environment_folder}/ART_kernel_specular.mtx')
art_reflect_matrix = scipy.io.mmread(f'{environment_folder}/ART_kernel_band_1.mtx')
art_patching_matrix = scipy.io.mmread(f'{environment_folder}/path_indexing.mtx')

# Convert to dense format if needed for visualization (optional)
art_dense_matrix = art_reflect_matrix.todense()
art_patching_matrix_dense = art_patching_matrix.todense()
art_dense_diffuse_matrix = art_matrix_diffuse.todense()
art_dense_spec_matrix = art_matrix_specular.todense()

num_non_zero_elems = np.count_nonzero(art_patching_matrix_dense)
assert art_dense_matrix.shape[0] == num_non_zero_elems

# Extract patch labels in mesh order from mesh.obj
mesh_obj_path = Path(environment_folder) / 'mesh.obj'
patch_labels = []
with open(mesh_obj_path, 'r', encoding='utf-8') as f:
    for line in f:
        line_no_comment = line.split('#', 1)[0].strip()
        if not line_no_comment.startswith('usemtl '):
            continue
        mat_name = line_no_comment.split()[1]
        if mat_name not in patch_labels:
            patch_labels.append(mat_name)

num_patches = art_patching_matrix_dense.shape[0]
assert len(patch_labels) == num_patches, (
    f'Expected {num_patches} patch labels from mesh.obj, found {len(patch_labels)}.'
)

#### Plot patching matrix - out of 256 total connections, 134 are valid

In [ ]:
# Create a Spy plot to visualize sparsity
plt.figure(figsize=(9, 8))
plt.spy(art_patching_matrix, markersize=2)
plt.title('Patching Matrix Sparsity Pattern')
plt.xlabel('Patch #')
plt.ylabel('Patch #')
plt.xticks(range(num_patches), patch_labels, rotation=90, fontsize=7)
plt.yticks(range(num_patches), patch_labels, fontsize=7)
plt.tight_layout()
plt.savefig(f'{fig_path.resolve()}/{environment_name}_patching_matrix.png')
plt.show()

#### Plot the sparse ART reflection matrix and its pruned, dense version for each patch

In [ ]:
# Create a Heatmap for density
plt.figure(figsize=(6, 6))
plt.imshow(art_dense_matrix, cmap='viridis', interpolation='nearest')
plt.colorbar()
plt.title('ART matrix')
plt.xlabel('Out. chans (idx of ij)')
plt.ylabel('Inc. chans (idx of hi)')
plt.show()

art_matrix_pruned_list = mm.prune_tdart_matrix(art_dense_matrix, art_patching_matrix.copy())
fig, axes = plt.subplots(4, 4, figsize=(10, 10), constrained_layout=True)
for i, ax in enumerate(axes.flat):
    # make sure no zero elements in the matrices
    num_elems =  art_matrix_pruned_list[i].shape[0] * art_matrix_pruned_list[i].shape[1]
    assert np.count_nonzero(art_matrix_pruned_list[i]) - num_elems == 0
    ax.imshow(art_matrix_pruned_list[i], cmap='viridis', interpolation='nearest')
    ax.set_title(f"{patch_labels[i]}", fontsize=8)
    ax.set_xlabel("i -> j")
    ax.set_ylabel("h -> i")
    
plt.show()

#### Plot the doubly stochastic Sinkhorn Knopp versions of the smaller matrices

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10), constrained_layout=True)
art_matrix_pruned_double_stochastic = []

for i, ax in enumerate(axes.flat):
    try:
        cur_double_stochastic = mm.sinkhornKnopp(art_matrix_pruned_list[i], verbose=False)
    except AssertionError as e:
        logger.error("Sinkhorn Knopp failed")
    art_matrix_pruned_double_stochastic.append(cur_double_stochastic)
    assert np.allclose(np.sum(art_matrix_pruned_double_stochastic[i], axis=0), np.ones(art_matrix_pruned_double_stochastic[i].shape[0]))
    assert np.allclose(np.sum(art_matrix_pruned_double_stochastic[i], axis=1), np.ones(art_matrix_pruned_double_stochastic[i].shape[1]))
    ax.imshow(art_matrix_pruned_list[i], cmap='viridis', interpolation='nearest')
    ax.set_title(f"{patch_labels[i]}", fontsize=8)
    ax.set_xlabel("i -> j")
    ax.set_ylabel("h -> i")

#### Find the FIR PU matrix whose average power gives the desired doubly stochastic matrix

This is the most challenging part, 
   - We do a decomposition: $\hat{\mathbf{S}}_p = P_0 T_1 T_2 \ldots T_K P_{K+1}$, where $P_k$ is a permutation matrix, and $T_r$ is identity except on a $2 \times 2$ block, $\begin{bmatrix} t_r & 1-t_r \\ 1-t_r & t_r\end{bmatrix}, 0 \leq t_r \leq 1$.  Now, $T_r = |G_r|^{\odot 2}$ where $G_r$ is a Givens rotation matrix.
   - Now, we can build $\mathbf{H}(z) = P_0 G_1 D_1(z) G_2 D_2(z) \ldots G_K P_{K+1}$, where $D_r(z) = \text{diag}(z^{d_{r,1}}, \ldots, z^{d_{r,N-k}})$.
   - Note that, $\frac{1}{2 \pi} \int_0^{2\pi}|\mathbf{H}(e^{j\omega})|^{\odot 2} d\omega  = P_0 |G_1|^{\odot 2} |G_2|^{\odot 2} \ldots |G_{K}|^{\odot 2} = \hat{\mathbf{S}}_p$. This holds as long as the delays $d_{r, n}$ are all pairwise distinct.
   - This is because $H_{ij}(e^{j\omega}) = \sum_p \alpha_p e^{-j\omega \tau_p}$, where $\tau_p$ is the total accumulated delay along that path, i.e, $\tau_p = \sum_{k=1}^K \omega_{k, c_k(p)}$ and $\alpha_p$ is the product of the Givens coefficients along that path. Here, $c_k(p)$ is a path such that $c_0(p) = j$, and $c_K(p) = i$. Therefore, $\frac{1}{2 \pi} \int_0^{2\pi}|H_{ij}(e^{j\omega})|^2 d\omega = \sum_p |\alpha_p|^2 + \sum_{p \neq q} \alpha_p \alpha_q \frac{1}{2 \pi} \int_{0}^{2\pi} e^{-j\omega(\tau_p - \tau_q)} d\omega.$ The cross-terms vanish if $\tau_p \neq \tau_q$.
   - A sufficient condition is to choose $D_k(z) = \text{diag}(1, \ldots, z^{-2^{k-1}}, \ldots, 1)$. More generally, $d_{r, i} > \sum_{\ell=1}^{r-1} \max_{m} d_{\ell, m}$. This means: every delay used at stage r is larger than the largest possible total delay that could have accumulated from all previous stages.